<a href="https://colab.research.google.com/github/Ilhamlafeer/Airline_Sentiment_Analysis_Using_BERT/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

MODEL_PATH = "/content/drive/MyDrive/NLP/airline_sentiment_bert"

print(os.listdir(MODEL_PATH))

['label_mapping.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [3]:
!pip install -q streamlit transformers torch

In [4]:
%%writefile app.py

import json
import os

import streamlit as st
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# PAGE CONFIG
st.set_page_config(
    page_title="Airline Sentiment Analysis",
    page_icon="✈️",
    layout="centered"
)


# MODEL PATH
MODEL_PATH = (
    "/content/drive/MyDrive/NLP/airline_sentiment_bert"
)


# LOAD MODEL
@st.cache_resource
def load_model():

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        local_files_only=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        local_files_only=True
    )

    model.eval()

    return tokenizer, model


# LOAD LABEL MAPPING
@st.cache_data
def load_labels():

    label_path = os.path.join(
        MODEL_PATH,
        "label_mapping.json"
    )

    with open(label_path, "r") as f:
        return json.load(f)


# LOAD MODEL
try:

    tokenizer, model = load_model()
    label_mapping = load_labels()

except Exception as e:

    st.error("Model loading failed.")

    st.exception(e)

    st.stop()


# PREDICTION
def predict_sentiment(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[
        0,
        predicted_class
    ].item()

    sentiment = label_mapping[
        str(predicted_class)
    ]

    return sentiment, confidence, probabilities


# TITLE
st.title("Airline Sentiment Analysis")

st.write(
    "BERT-based sentiment classification for airline-related tweets."
)

st.divider()


# TEXT INPUT
st.subheader("Enter an Airline Tweet")

tweet = st.text_area(
    "Tweet",
    placeholder=(
        "Example: The flight attendant was very "
        "helpful and friendly!"
    ),
    height=150,
    label_visibility="collapsed"
)


# BUTTON

if st.button(
    "🔍 Analyze Sentiment",
    use_container_width=True
):

    if not tweet.strip():

        st.warning(
            "Please enter a tweet."
        )

    else:

        with st.spinner(
            "Analyzing..."
        ):

            sentiment, confidence, probabilities = (
                predict_sentiment(tweet)
            )

        st.divider()

        # MAIN RESULT
        st.subheader("Prediction")

        if sentiment.lower() == "positive":

            st.success(
                f"😊 POSITIVE\n\n"
                f"Confidence: {confidence:.2%}"
            )

        elif sentiment.lower() == "negative":

            st.error(
                f"😠 NEGATIVE\n\n"
                f"Confidence: {confidence:.2%}"
            )

        else:

            st.info(
                f"😐 NEUTRAL\n\n"
                f"Confidence: {confidence:.2%}"
            )


        # PROBABILITIES
        st.subheader(
            "Sentiment Probabilities"
        )

        for class_id, class_name in label_mapping.items():

            probability = probabilities[
                0,
                int(class_id)
            ].item()

            st.write(
                f"**{class_name.capitalize()}**: "
                f"{probability:.2%}"
            )

            st.progress(
                float(probability)
            )


# SIDEBAR
with st.sidebar:

    st.header("About")

    st.write(
        """
        This application uses a fine-tuned
        BERT model to classify airline tweets
        into three sentiment categories.
        """
    )

    st.divider()

    st.subheader("Classes")

    st.write("😠 Negative")
    st.write("😐 Neutral")
    st.write("😊 Positive")

    st.divider()

    st.subheader("Model Information")

    st.write(
        "**Base Model:** BERT-base-uncased"
    )

    st.write(
        "**Task:** 3-class classification"
    )

    st.write(
        "**Max Sequence Length:** 128"
    )

    st.write(
        "**Test Accuracy:** 84.95%"
    )

Overwriting app.py


# Streamlit app

In [5]:
!streamlit run app.py &> /content/streamlit.log &

In [7]:
!cat /content/streamlit.log



2026-08-16 07:43:54.832 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.34.97.242:8501



# Create a public URL (Install Cloudflare Tunnel)

In [8]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 118336 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.8.2) over (2026.8.2) ...
Setting up cloudflared (2026.8.2) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-08-16T07:44:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-16T07:44:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-16T07:44:21Z INF +--------------------------------------------------------------------------------------------+
2026-08-16T07:44:21Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-16T07:44:21Z INF |  https://quote-candle-rand-losing.trycloudflare.com   